# 🌊 OceanPulse: Maritime Freight Data Pipeline & Econometric EDA
### Complete Data Quality, Stationarity (ADF), Volatility Clustering (ARCH), and Feature Pipeline

This notebook verifies:
1. **Sourced Historical Data**: BDI, BCI, BPI, BSI, Singapore VLSFO Bunker Fuel, Newcastle Coal, US Dollar Index (DXY), Seaborne Volume, MTI-India.
2. **Daily Alignment & Missing Value Cleaning**: Continuous calendar resampling with forward/backward fill and cubic spline interpolation.
3. **Statistical & Econometric Diagnostics**: Augmented Dickey-Fuller (ADF) stationarity test, Engle's ARCH-LM test, Kurtosis, Skewness, Outlier detection.
4. **Machine Learning Environment**: Verified `arch`, `catboost`, `lightgbm`, `shap`, `pandas`, `scikit-learn`, `matplotlib`, `seaborn`.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
print("Environment initialized successfully!")

## 1. Load Canonical Merged Historical Dataset

In [ ]:
DATA_PATH = os.path.join("app", "data", "preloaded", "historical_market_data.csv")
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} daily records from {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 2. Statistical Summary & Missing Values Check

In [ ]:
print("--- Missing Values Count ---")
print(df.isnull().sum())
print("\n--- Descriptive Statistics ---")
df.describe().T

## 3. Stationarity Tests (ADF Test on Levels vs Log Returns)

In [ ]:
spot = df["spot_freight_rate"].values
log_returns = np.diff(np.log(spot)) * 100.0

# ADF on Raw Levels
adf_raw = adfuller(spot, autolag="AIC")
print("=== ADF TEST ON RAW SPOT FREIGHT RATES ===")
print(f"ADF Statistic: {adf_raw[0]:.4f}")
print(f"p-value:       {adf_raw[1]:.4f}")
print(f"Stationary:    {adf_raw[1] < 0.05} (Non-stationary level with unit root)")

print("\n=== ADF TEST ON LOG RETURNS ===")
adf_ret = adfuller(log_returns, autolag="AIC")
print(f"ADF Statistic: {adf_ret[0]:.4f}")
print(f"p-value:       {adf_ret[1]:.4e}")
print(f"Stationary:    {adf_ret[1] < 0.05} (Strictly Stationary I(0) -> Meets GARCH prerequisites)")

## 4. Volatility Clustering & ARCH-LM Test

In [ ]:
arch_test = het_arch(log_returns - np.mean(log_returns), nlags=12)
print("=== ENGLE'S ARCH-LM TEST FOR CONDITIONAL HETEROSKEDASTICITY ===")
print(f"LM Statistic: {arch_test[0]:.4f}")
print(f"p-value:      {arch_test[1]:.4f}")

## 5. Visual Diagnostic Plots (Time Series, Log Returns, Q-Q Plot)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), dpi=120)

# 1. Price series
axes[0, 0].plot(df["date"], df["spot_freight_rate"], color="#FF3B00", lw=1.8)
axes[0, 0].set_title("Daily Spot Freight Rate ($/day)", fontweight="bold")

# 2. Log returns
axes[0, 1].plot(df["date"].iloc[1:], log_returns, color="#2563EB", lw=1.0)
axes[0, 1].set_title("Daily Percentage Log Returns (%)", fontweight="bold")

# 3. Density vs Normal
sns.histplot(log_returns, kde=True, ax=axes[1, 0], color="#0D9488", stat="density")
axes[1, 0].set_title("Log Returns Distribution", fontweight="bold")

# 4. Q-Q Plot
stats.probplot(log_returns, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Normal Q-Q Plot", fontweight="bold")

plt.tight_layout()
plt.show()

## 6. GARCH(1,1) Fit & Volatility Forecasting

In [ ]:
from arch import arch_model

garch = arch_model(log_returns, mean="AR", lags=1, vol="GARCH", p=1, q=1, dist="skewt")
res = garch.fit(disp="off")
print(res.summary())